### 1 Downloading NDVI

In [ ]:
import ee
import geemap
import os

ee.Initialize()

# ======================
# ROI
# ======================
roi = ee.FeatureCollection(
    "projects/lithe-inscriber-448612-q7/assets/tekeze"
)

# ======================
# MODIS NDVI
# ======================
col = (ee.ImageCollection("MODIS/061/MOD13A3")
       .select("NDVI")
       .filterDate("2011-01-01", "2025-12-31")
       .filterBounds(roi)   # 🔥 ADD THIS
       .map(lambda img: img.multiply(0.0001)
            .clip(roi)
            .copyProperties(img, ["system:time_start"]))
)

# ======================
# Output folder in Drive
# ======================
folder = "GEE_NDVI_2011_2025_4326"

size = col.size().getInfo()
images = col.toList(size)

print("Total images:", size)

# ======================
# Export loop (EPSG:4326 FIXED)
# ======================
for i in range(size):

    img = ee.Image(images.get(i))

    date = ee.Date(img.get("system:time_start")).format("YYYY_MM").getInfo()

    filename = f"NDVI_{date}"

    task = ee.batch.Export.image.toDrive(
        image=img,
        description=filename,
        folder=folder,
        fileNamePrefix=filename,
        region=roi.geometry(),
        scale=1000,
        crs="EPSG:4326",   # 🔥 KEY FIX
        maxPixels=1e13
    )

    task.start()
    print("Started:", filename)

### Visualization 2010

In [ ]:
import os
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

# ======================
# Folder
# ======================
folder = r"C:\Users\gerii\Documents\projects\tekeze\NDVI_outputs"

files = sorted([
    f for f in os.listdir(folder)
    if f.startswith("NDVI_2010") and f.endswith(".tif")
])

# ======================
# Figure
# ======================
fig, axes = plt.subplots(3, 4, figsize=(14, 9))
axes = axes.flatten()

months = ["Jan","Feb","Mar","Apr","May","Jun",
          "Jul","Aug","Sep","Oct","Nov","Dec"]

# ======================
# NDVI range
# ======================
vmin = -0.2
vmax = 1.0

# ======================
# Plot loop
# ======================
for i, f in enumerate(files):

    path = os.path.join(folder, f)

    with rasterio.open(path) as src:
        data = src.read(1).astype("float32")

        # keep water (negative NDVI), remove extreme invalid
        data[data < -0.3] = np.nan

        bounds = src.bounds
        extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]

    ax = axes[i]

    im = ax.imshow(
        data,
        extent=extent,
        origin="upper",
        cmap="RdYlGn",
        vmin=vmin,
        vmax=vmax
    )

    ax.set_title(months[i], fontsize=10)

    # ======================
    # 🔥 1-degree tick interval (NEW)
    # ======================
    ax.xaxis.set_major_locator(MultipleLocator(1))
    ax.yaxis.set_major_locator(MultipleLocator(1))
    ax.tick_params(labelsize=8)

    # ======================
    # Axis visibility control
    # ======================
    row = i // 4
    col = i % 4

    # Latitude only on left
    if col == 0:
        ax.set_ylabel("Lat", fontsize=9)
    else:
        ax.set_yticklabels([])

    # Longitude only on bottom
    if row == 2:
        ax.set_xlabel("Lon", fontsize=9)
    else:
        ax.set_xticklabels([])

# ======================
# Remove unused axes
# ======================
for j in range(len(files), 12):
    axes[j].axis("off")

# ======================
# Tight layout spacing
# ======================
plt.subplots_adjust(
    left=0.06,
    right=0.88,
    top=0.92,
    bottom=0.08,
    wspace=0.02,
    hspace=0.05
)

# ======================
# External colorbar
# ======================
cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label("NDVI", fontsize=11)

# ======================
# Title
# ======================
plt.suptitle("Monthly NDVI Maps (2010)", fontsize=14)

# ======================
# Save (FIXED)
# ======================
fig.canvas.draw()
fig.savefig(
    r"C:\Users\gerii\Documents\projects\tekeze\NDVI_outputs\NDVI_2010_pub.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

### 3 compute VCI

In [ ]:
import os
import numpy as np
import rasterio

# ======================
# Input / Output folders
# ======================
in_folder = r"C:\Users\gerii\Documents\projects\tekeze\NDVI_outputs"
out_folder = r"C:\Users\gerii\Documents\projects\tekeze\VCI"

os.makedirs(out_folder, exist_ok=True)

# ======================
# Get all NDVI files
# ======================
files = sorted([
    f for f in os.listdir(in_folder)
    if f.startswith("NDVI_") and f.endswith(".tif")
])

print("Total NDVI files:", len(files))

# ======================
# Read all images into stack
# ======================
stack = []
meta = None

for f in files:
    path = os.path.join(in_folder, f)

    with rasterio.open(path) as src:
        data = src.read(1).astype("float32")

        # Handle invalid values
        data[data < -0.3] = np.nan

        stack.append(data)

        if meta is None:
            meta = src.meta.copy()

stack = np.array(stack)  # shape = (time, rows, cols)

print("Stack shape:", stack.shape)

# ======================
# Compute per-pixel min & max
# ======================
ndvi_min = np.nanmin(stack, axis=0)
ndvi_max = np.nanmax(stack, axis=0)

# Avoid division by zero
den = ndvi_max - ndvi_min
den[den == 0] = np.nan

print("Min/Max computed ✔")

# ======================
# Update metadata for output
# ======================
meta.update(dtype="float32", count=1)

# ======================
# Compute and save VCI
# ======================
for i, f in enumerate(files):

    ndvi = stack[i]

    # VCI formula
    vci = (ndvi - ndvi_min) / den

    # Clip to valid range
    vci = np.clip(vci, 0, 1)

    # Convert to percentage (optional but standard)
    #vci = vci * 100

    # Output filename
    out_name = f.replace("NDVI_", "VCI_")
    out_path = os.path.join(out_folder, out_name)

    # Save GeoTIFF
    with rasterio.open(out_path, "w", **meta) as dst:
        dst.write(vci.astype("float32"), 1)

    print("Saved:", out_name)

print("All VCI files created successfully ✔")

### 4 Visulize VCI 2010

In [ ]:
import os
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

# ======================
# Folder (VCI outputs)
# ======================
folder = r"C:\Users\gerii\Documents\projects\tekeze\VCI"

files = sorted([
    f for f in os.listdir(folder)
    if f.startswith("VCI_2010") and f.endswith(".tif")
])

print("Files found:", len(files))

# ======================
# Figure
# ======================
fig, axes = plt.subplots(3, 4, figsize=(14, 9))
axes = axes.flatten()

months = ["Jan","Feb","Mar","Apr","May","Jun",
          "Jul","Aug","Sep","Oct","Nov","Dec"]

# ======================
# VCI range (FIXED)
# ======================
vmin = 0
vmax = 1

# ======================
# Plot
# ======================
for i, f in enumerate(files):

    path = os.path.join(folder, f)

    with rasterio.open(path) as src:
        data = src.read(1).astype("float32")

        # mask invalid
        data[(data < 0) | (data > 1)] = np.nan

        bounds = src.bounds
        extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]

    ax = axes[i]

    im = ax.imshow(
        data,
        extent=extent,
        origin="upper",
        cmap="YlGn",   # low=yellow, high=green
        vmin=vmin,
        vmax=vmax
    )

    ax.set_title(months[i], fontsize=10)

    # ======================
    # 1-degree grid spacing
    # ======================
    ax.xaxis.set_major_locator(MultipleLocator(1))
    ax.yaxis.set_major_locator(MultipleLocator(1))
    ax.tick_params(labelsize=8)

    # ======================
    # Axis styling
    # ======================
    row = i // 4
    col = i % 4

    if col == 0:
        ax.set_ylabel("Lat", fontsize=9)
    else:
        ax.set_yticklabels([])

    if row == 2:
        ax.set_xlabel("Lon", fontsize=9)
    else:
        ax.set_xticklabels([])

# ======================
# Remove empty axes
# ======================
for j in range(len(files), 12):
    axes[j].axis("off")

# ======================
# Tight layout
# ======================
plt.subplots_adjust(
    left=0.06,
    right=0.88,
    top=0.92,
    bottom=0.08,
    wspace=0.02,
    hspace=0.05
)

# ======================
# Colorbar (FIXED)
# ======================
cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label("VCI", fontsize=11)

# ======================
# Title
# ======================
plt.suptitle("Monthly Vegetation Condition Index (VCI) - 2010", fontsize=14)

# ======================
# Save figure
# ======================
fig.canvas.draw()
fig.savefig(
    r"C:\Users\gerii\Documents\projects\tekeze\VCI\VCI_2010_map.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()